# OpenAI Capability Smoke Test

This notebook wires the `AgentBuilder` into the OpenAI LLM adapter so you can
add capabilities like context packs and memory "just like lego pieces". Set
`OPENAI_API_KEY` in your environment (or run the helper cell below) before
executing the agent cell.

In [ ]:
import os
from getpass import getpass

from agentic_codex import AgentBuilder, Context, EnvOpenAIAdapter
from agentic_codex.core.memory import EpisodicMemory
from agentic_codex.core.schemas import AgentStep, Message

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY: ")  # noqa: S603

In [ ]:
KNOWLEDGE_BASE = [
    {
        "topic": "orion",
        "answer": "Orion is a prominent constellation located on the celestial equator.",
    },
    {
        "topic": "andromeda galaxy",
        "answer": "The Andromeda Galaxy is the nearest major galaxy to the Milky Way.",
    },
]


def query_step(ctx: Context) -> AgentStep:
    question = ctx.goal
    context_pack = ctx.components["context"]
    knowledge = context_pack.get("knowledge", KNOWLEDGE_BASE)
    answer = next(
        (record["answer"] for record in knowledge if record["topic"] in question.lower()),
        "I could not find that in the provided context.",
    )

    memory = ctx.components["memory"]
    memory.put("question", question)
    memory.put("retrieved_answer", answer)

    if ctx.llm is None:
        final_answer = answer
    else:
        prompt = (
            "You are a helpful assistant. Answer the question using the provided context.\n"
            f"Question: {question}\n"
            f"Context: {answer}\n"
            "Answer:"
        )
        final_answer = ctx.llm.generate(prompt)

    memory.put("final_answer", final_answer)
    return AgentStep(out_messages=[Message(role="assistant", content=final_answer)])

In [ ]:
builder = (
    AgentBuilder(name="openai-query", role="assistant")
    .with_llm(EnvOpenAIAdapter(model="gpt-4o-mini"))
    .with_context({"knowledge": KNOWLEDGE_BASE})
    .with_memory(EpisodicMemory())
    .with_step(query_step)
)

agent = builder.build()
ctx = Context(goal="What is the Andromeda Galaxy?")
result = agent.run(ctx)
result.out_messages[-1].content